In [1]:
import string
import torch
import numpy as np
from tqdm import tqdm
import torch.nn as nn
import albumentations as A
import torch.nn.functional as F
import glob , json , timm , cv2 , random
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from torchsummary import summary
from PIL import Image, ImageDraw, ImageFont

device = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"device: {device}")

device: cuda


### Datasets

In [2]:

IMG_W = 256  # Expanded width for 32 time-steps
IMG_H = 32

CHARS = (
    string.digits +
    string.ascii_uppercase +
    string.ascii_lowercase
)

FONTS = [
    cv2.FONT_HERSHEY_SIMPLEX,
    cv2.FONT_HERSHEY_DUPLEX,
    cv2.FONT_HERSHEY_COMPLEX,
    cv2.FONT_HERSHEY_TRIPLEX,
]


# --- Load the Real-World Fonts ---
# Make sure you have a 'fonts' directory with some .ttf files in it!
FONT_PATHS = glob.glob("fonts/*/*.[to]tf") 
if not FONT_PATHS:
    raise RuntimeError("No fonts found! Please add some .ttf files to a 'fonts' directory.")

def random_text():
    length = random.randint(3, 12)
    return "".join(random.choice(CHARS) for _ in range(length))

def render_text_pil(text):
    """
    Renders text using real typography (anti-aliased, TrueType) via PIL,
    then converts it back to an OpenCV numpy array for augmentations.
    """
    # 1. Create a pure black canvas
    img_pil = Image.new('L', (IMG_W, IMG_H), color=0)
    draw = ImageDraw.Draw(img_pil)
    
    # 2. Pick a random font and scale it 
    # (Sizes between 18 and 28 usually fit well in a 32px height)
    font_path = random.choice(FONT_PATHS)
    font_size = random.randint(18, 28)
    font = ImageFont.truetype(font_path, font_size)
    
    # 3. Calculate text bounding box to center it
    # textbbox is the modern PIL method for precise measurements
    bbox = draw.textbbox((0, 0), text, font=font)
    tw = bbox[2] - bbox[0]
    th = bbox[3] - bbox[1]
    
    # Add minor random jitter to X and Y
    x = max(0, (IMG_W - tw) // 2) + random.randint(-8, 8)
    y = max(0, (IMG_H - th) // 2) + random.randint(-2, 2)
    
    # 4. Draw the text in pure white (255)
    draw.text((x, y), text, font=font, fill=255)
    
    # 5. Convert back to OpenCV (numpy) format
    img_cv = np.array(img_pil)
    
    # 6. Optional: Simulate slight ink bleed/bolding
    if random.random() < 0.3:
        kernel = np.ones((2, 2), np.uint8)
        img_cv = cv2.dilate(img_cv, kernel, iterations=1)
        
    return img_cv

def augment(img):
    if random.random() < 0.3:
        pts1 = np.float32([[0, 0], [IMG_W, 0], [0, IMG_H], [IMG_W, IMG_H]])
        diff = np.random.uniform(-4, 4, size=(4, 2)).astype(np.float32)
        pts2 = pts1 + diff
        matrix = cv2.getPerspectiveTransform(pts1, pts2)
        img = cv2.warpPerspective(img, matrix, (IMG_W, IMG_H), borderValue=0)

    if random.random() < 0.5:
        angle = random.uniform(-5, 5)
        M = cv2.getRotationMatrix2D((IMG_W // 2, IMG_H // 2), angle, 1.0)
        img = cv2.warpAffine(img, M, (IMG_W, IMG_H), borderValue=0)

    if random.random() < 0.3:
        kernel = np.ones((2, 2), np.uint8)
        if random.random() < 0.5:
            img = cv2.erode(img, kernel, iterations=1)
        else:
            img = cv2.dilate(img, kernel, iterations=1)

    if random.random() < 0.2:
        k = random.choice([3, 5])
        kernel = np.zeros((k, k))
        kernel[int((k-1)/2), :] = np.ones(k)
        kernel /= k
        img = cv2.filter2D(img, -1, kernel)
    elif random.random() < 0.4:
        img = cv2.GaussianBlur(img, (3, 3), 0)

    if random.random() < 0.2:
        pt1 = (random.randint(0, IMG_W), random.randint(0, IMG_H))
        pt2 = (random.randint(0, IMG_W), random.randint(0, IMG_H))
        cv2.line(img, pt1, pt2, 255, random.randint(1, 2))

    if random.random() < 0.5:
        img = cv2.bitwise_not(img)

    if random.random() < 0.4:
        noise = np.random.normal(0, random.uniform(5, 15), img.shape)
        img = np.clip(img.astype(np.float32) + noise, 0, 255).astype(np.uint8)

    if random.random() < 0.3:
        alpha = random.uniform(0.7, 1.3)
        img = np.clip(img.astype(np.float32) * alpha, 0, 255).astype(np.uint8)

    return img

def generate_sample():
    text = random_text()
    # --- Swap in the PIL renderer here ---
    img = render_text_pil(text)
    # Your existing augmentations work perfectly on the new numpy array
    img = augment(img) 
    img = img.astype(np.float32) / 255.0
    img = np.stack([img, img, img], axis=0)
    return img, text

char_to_idx = {c: i + 1 for i, c in enumerate(CHARS)}
idx_to_char = {i + 1: c for i, c in enumerate(CHARS)}
BLANK_IDX = 0


# Character set configuration
CHARS = string.digits + string.ascii_uppercase + string.ascii_lowercase

def random_text():
    length = random.randint(3, 12)
    return "".join(random.choice(CHARS) for _ in range(length))

class HybridOCRDataset(Dataset):
    def __init__(self, real_data_list, img_w=256, img_h=32, epoch_size=100000):
        self.real_data = real_data_list
        self.img_w = img_w
        self.img_h = img_h
        self.epoch_size = epoch_size

        # 1. Load Fonts
        self.font_list = glob.glob("./fonts/*/*.ttf") + glob.glob("./fonts/*/*.otf")
        if not self.font_list:
            print("WARNING: No .ttf or .otf files found in ./fonts/ directory!")

        # 2. Initialize OCR-Safe Augmentations
        self.ocr_transform = A.Compose([
            A.MotionBlur(blur_limit=5, p=0.3),
            A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),
            A.GridDistortion(num_steps=5, distort_limit=0.2, p=0.3)
        ])

        # 3. Curriculum State Variables
        self.phase = 1 
        self.use_augmentations = False
        self.real_data_prob = 0.0
        self.use_stencils = False

    def set_curriculum_phase(self, phase):
        """Dynamically adjusts the difficulty of the dataset."""
        self.phase = phase
        if phase == 1:
            self.use_augmentations = False
            self.real_data_prob = 0.0
            self.use_stencils = False
            print("\n--- ENTERING PHASE 1: CLEAN SYNTHETIC DATA ---")
        elif phase == 2:
            self.use_augmentations = True
            self.real_data_prob = 0.1
            self.use_stencils = False
            print("\n--- ENTERING PHASE 2: AUGMENTATIONS + REAL DATA ---")
        elif phase == 3:
            self.use_augmentations = True
            self.real_data_prob = 0.2
            self.use_stencils = True
            print("\n--- ENTERING PHASE 3: MAX DISTORTION + STENCILS ---")

    def __len__(self):
        return self.epoch_size 

    def __getitem__(self, idx):
        # Dynamically apply real data probability based on current Phase
        if random.random() < self.real_data_prob and len(self.real_data) > 0:
            img_path, text = random.choice(self.real_data)
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            
            if img is None:
                img_tensor, text = self._get_synthetic()
            else:
                if self.use_augmentations:
                    augmented_img = self.ocr_transform(image=img)['image']
                    img_tensor = self._preprocess_real(augmented_img)
                else:
                    img_tensor = self._preprocess_real(img)
        else:
            img_tensor, text = self._get_synthetic()

        return torch.tensor(img_tensor, dtype=torch.float32), text

    def _get_synthetic(self):
        text = random_text() 
        
        # Select Font
        font_path = random.choice(self.font_list)
        font_size = random.randint(28, 36)
        font = ImageFont.truetype(font_path, font_size)
        
        # Render Text
        image = Image.new('L', (self.img_w, self.img_h), color=255)
        draw = ImageDraw.Draw(image)
        
        bbox = draw.textbbox((0, 0), text, font=font)
        w, h = bbox[2] - bbox[0], bbox[3] - bbox[1]
        x, y = (self.img_w - w) / 2, (self.img_h - h) / 2
        
        draw.text((x, y), text, font=font, fill=0)
        img_np = np.array(image)

        # Apply augmentations if Phase is > 1
        if self.use_augmentations:
            img_np = self.ocr_transform(image=img_np)['image']
        
        img_tensor = img_np.astype(np.float32) / 255.0
        return np.stack([img_tensor, img_tensor, img_tensor], axis=0), text

    def _preprocess_real(self, gray_img):
        h, w = gray_img.shape
        scale = min(self.img_w / w, self.img_h / h)
        new_w, new_h = max(1, int(w * scale)), max(1, int(h * scale))
        
        resized = cv2.resize(gray_img, (new_w, new_h), interpolation=cv2.INTER_AREA)
        
        bg_color = int(np.median(gray_img))
        canvas = np.full((self.img_h, self.img_w), bg_color, dtype=np.uint8)
        
        x_off, y_off = (self.img_w - new_w) // 2, (self.img_h - new_h) // 2
        canvas[y_off:y_off+new_h, x_off:x_off+new_w] = resized
        
        canvas = canvas.astype(np.float32) / 255.0
        return np.stack([canvas, canvas, canvas], axis=0)

def collate_fn(batch):
    images, texts = zip(*batch)
    
    images = torch.stack(images)
    targets = []
    target_lengths = []
    
    for text in texts:
        encoded = [char_to_idx[c] for c in text if c in char_to_idx]
        targets.extend(encoded)
        # CRITICAL FIX: Use len(encoded) instead of len(text)
        target_lengths.append(len(encoded)) 
        
    targets = torch.tensor(targets, dtype=torch.long)
    target_lengths = torch.tensor(target_lengths, dtype=torch.long)
    
    return images, targets, target_lengths, texts



def ctc_decode(logits):
    pred = logits.argmax(-1)
    text = []
    prev = -1
    for p in pred:
        p = p.item()
        if p != prev and p != 0:
            text.append(idx_to_char[p])
        prev = p
    return "".join(text)

def levenshtein_distance(s1, s2):
    if len(s1) > len(s2):
        s1, s2 = s2, s1
    distances = range(len(s1) + 1)
    for i2, c2 in enumerate(s2):
        distances_ = [i2+1]
        for i1, c1 in enumerate(s1):
            if c1 == c2:
                distances_.append(distances[i1])
            else:
                distances_.append(1 + min((distances[i1], distances[i1 + 1], distances_[-1])))
        distances = distances_
    return distances[-1]

### Model Definition

In [3]:
class EnhancedOCRNet(nn.Module):
    def __init__(self, num_chars, dropout_p=0.2):
        super().__init__()
        
        # 1. Doubled CNN filters (050 -> 100) to resolve micro-features
        self.backbone = timm.create_model(
            "mobilenetv3_small_100", 
            pretrained=True, 
            features_only=True, 
            out_indices=(2,) # MUST stay (2,) to maintain 32 timesteps
        )
        
        with torch.no_grad():
            dummy_input = torch.zeros(1, 3, 32, 256)
            out_channels = self.backbone(dummy_input)[0].shape[1]
        
        # 2. Moderately increased RNN capacity (128 -> 160)
        self.rnn = nn.GRU(
            input_size=out_channels, 
            hidden_size=160,       
            num_layers=2,  
            bidirectional=True,
            batch_first=True,
            dropout=dropout_p
        )
        
        # 3. Scaled head to match new BiGRU output (160 * 2 = 320)
        self.head = nn.Sequential(
            nn.Linear(320, 256),   
            nn.Hardswish(inplace=True),
            nn.LayerNorm(256),
            nn.Dropout(dropout_p),
            nn.Linear(256, num_chars + 1)
        )

    def forward(self, x):
        x = self.backbone(x)[0]
        x = x.mean(dim=2)
        x = x.permute(0, 2, 1)
        x, _ = self.rnn(x)  
        x = self.head(x)  
        return x

In [4]:
from torchinfo import summary
model = EnhancedOCRNet(num_chars=len(CHARS))
model.cpu() 
dummy_input = torch.zeros(1, 3, 32, 256)
summary(model, input_data=dummy_input,col_names=["input_size", "output_size", "num_params", "trainable"],col_width=20,row_settings=["var_names"])

Unexpected keys (classifier.bias, classifier.weight, conv_head.bias, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.


Layer (type (var_name))                                 Input Shape          Output Shape         Param #              Trainable
EnhancedOCRNet (EnhancedOCRNet)                         [1, 3, 32, 256]      [1, 32, 63]          --                   True
├─MobileNetV3Features (backbone)                        [1, 3, 32, 256]      [1, 24, 4, 32]       --                   True
│    └─Conv2d (conv_stem)                               [1, 3, 32, 256]      [1, 16, 16, 128]     432                  True
│    └─BatchNorm2d (bn1)                                [1, 16, 16, 128]     [1, 16, 16, 128]     32                   True
│    └─Hardswish (act1)                                 [1, 16, 16, 128]     [1, 16, 16, 128]     --                   --
│    └─Sequential (blocks)                              --                   --                   --                   True
│    │    └─Sequential (0)                              [1, 16, 16, 128]     [1, 16, 8, 64]       744                  True
│    

### Model Training

In [5]:
### Loss func
class FocalCTCLoss(nn.Module):
    def __init__(self, blank=0, alpha=1.0, gamma=2.0):
        super().__init__()
        self.ctc = nn.CTCLoss(blank=blank, zero_infinity=True, reduction='none')
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, log_probs, targets, input_lengths, target_lengths):
        loss = self.ctc(log_probs, targets, input_lengths, target_lengths)
        
        # CRITICAL SHIELD: Clamping prevents negative garbage values 
        # from triggering an exponential math explosion.
        loss = torch.clamp(loss, min=0.0)
        
        p = torch.exp(-loss)
        focal_loss = self.alpha * ((1 - p) ** self.gamma) * loss
        
        return focal_loss.mean()



with open("train_labels.json", "r") as f:
    real_data = json.load(f)
unified_real_list = [(item["path"], item["label"]) for item in real_data]

train_dataset = HybridOCRDataset(real_data_list=unified_real_list, epoch_size=1024000)
train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=True, num_workers=8 , collate_fn=collate_fn)
val_dataset = HybridOCRDataset([], epoch_size=1024)
val_loader = DataLoader(val_dataset, batch_size=1024, shuffle=False, num_workers=8, collate_fn=collate_fn)

model = EnhancedOCRNet(num_chars=len(CHARS)).to(device)
## ctc_loss = nn.CTCLoss(blank=0, zero_infinity=True)
ctc_loss = FocalCTCLoss(blank=0)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

/tmp/ipykernel_32383/1380287823.py:150: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),
Unexpected keys (classifier.bias, classifier.weight, conv_head.bias, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.


In [6]:
EPOCHS = 500
ACCUMULATION_STEPS = 1
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_char_acc = 0.0
patience = 0

# Start in Phase 1 (Clean Data)
train_dataset.set_curriculum_phase(1)

for epoch in range(EPOCHS):
    # --- DYNAMIC DIFFICULTY ADJUSTMENT ---
    # Scaled for 500 epochs
    if epoch == 150:
        train_dataset.set_curriculum_phase(2)
    elif epoch == 350:
        train_dataset.set_curriculum_phase(3)

    model.train()
    running_loss = 0.0
    optimizer.zero_grad() # Zero gradients ONCE before the batch loop
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}")
    
    for step, (images, targets, target_lengths, _) in enumerate(pbar):
        images = images.to(device)
        targets = targets.to(device)
        target_lengths = target_lengths.to(device)

        logits = model(images)
        logits = logits.permute(1, 0, 2)
        log_probs = F.log_softmax(logits, dim=2)

        T, B = log_probs.size(0), log_probs.size(1)
        input_lengths = torch.full(size=(B,), fill_value=T, dtype=torch.long, device=device)
        
        # Divide loss by accumulation steps so the scaled gradient is correct
        loss = ctc_loss(log_probs, targets, input_lengths, target_lengths) / ACCUMULATION_STEPS
        loss.backward()

        # --- GRADIENT ACCUMULATION BLOCK ---
        if (step + 1) % ACCUMULATION_STEPS == 0 or (step + 1) == len(train_loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
            optimizer.zero_grad()

        # Multiply by ACCUMULATION_STEPS for logging purposes to see true loss
        running_loss += (loss.item() * ACCUMULATION_STEPS) 
        pbar.set_postfix(loss=loss.item() * ACCUMULATION_STEPS)

    # --- END OF EPOCH ROUTINE ---
    scheduler.step()
    epoch_loss = running_loss / len(train_loader)
    
    model.eval()
    total_edit_distance = 0
    total_gt_chars = 0
    
    with torch.no_grad():
        for val_images, _, _, val_texts in val_loader:
            val_images = val_images.to(device)
            val_logits = model(val_images)
            
            for idx in range(val_images.size(0)):
                pred_text = ctc_decode(val_logits[idx])
                gt_text = val_texts[idx]
                
                total_edit_distance += levenshtein_distance(pred_text, gt_text)
                total_gt_chars += max(len(gt_text), 1)
                
    char_accuracy = (1.0 - (total_edit_distance / total_gt_chars)) * 100
    print(f"Epoch {epoch} Summary: Loss={epoch_loss:.4f} | Char Accuracy={char_accuracy:.2f}%")
    
    # --- CHECKPOINTING & EARLY STOPPING ---
    if char_accuracy > best_char_acc:
        best_char_acc = char_accuracy
        checkpoint = {
            'epoch': epoch,
            'model_state': model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'best_acc': best_char_acc
        }
        torch.save(checkpoint, "best_ocr.pth")
        print(f"Saved Checkpoint with Char Accuracy: {best_char_acc:.2f}%")
        patience = 0
    else:
        patience += 1  # <-- FIXED

    if patience >= 50:
        print(f"Early stopping triggered at epoch {epoch}")
        break


--- ENTERING PHASE 1: CLEAN SYNTHETIC DATA ---


Epoch 0/500: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [04:41<00:00,  3.55it/s, loss=3.34]


Epoch 0 Summary: Loss=7.2889 | Char Accuracy=80.59%
Saved Checkpoint with Char Accuracy: 80.59%


Epoch 1/500: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [04:15<00:00,  3.91it/s, loss=2.58]


Epoch 1 Summary: Loss=2.8709 | Char Accuracy=49.88%


...

Epoch 82 Summary: Loss=1.8550 | Char Accuracy=87.81%


Epoch 83/500: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [04:15<00:00,  3.91it/s, loss=1.82]


Epoch 83 Summary: Loss=1.8628 | Char Accuracy=85.38%


Epoch 84/500:  25%|█████████████████████████████▌                                                                                        | 250/1000 [01:27<04:23,  2.85it/s, loss=1.91]


KeyboardInterrupt: 

### Evaluation of model

In [ ]:
model.eval()
print("\n--- Final Model Evaluation ---")
for i in range(10):
    img, gt = generate_sample()
    x = torch.tensor(img).float().unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(x)[0]
    pred = ctc_decode(logits)
    
    print(f"GT  : {gt}")
    print(f"PRED: {pred}")